## full fine tune BART for summarization

In [ ]:
!pip install transformers evaluate transformers[torch]
!pip install -U datasets

In [ ]:
# Load model
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
"""
BART HAS 400M PARAMS: https://github.com/facebookresearch/fairseq/tree/main/examples/bart
"""
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-large-cnn")
model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-large-cnn")

In [ ]:
!pip install py7zr

In [ ]:
# Load dataset
from datasets import load_dataset
dataset = load_dataset("ingeniumacademy/samsum")
dataset

In [ ]:
sample_data = dataset['test'][0]['dialogue']
label = dataset['test'][0]['summary']

def generate_summary(input_data, llm):
  input_prompt = f"""
                  Summarize the following conversation.

                  {input_data}

                  Summary:
                  """

  input_ids = tokenizer(sample_data, return_tensors='pt')
  tokenized_output = llm.generate(input_ids['input_ids'], min_length=30, max_length=200)
  output = tokenizer.decode(tokenized_output[0], skip_special_tokens=True)

  return output

output = generate_summary(sample_data, llm=model)
print("Sample data : ")
print(sample_data)
print("-------------------")
print("Model Generated Summary :")
print(output)
print("Correct Summary :")
print(label)

###prepare dataset

In [ ]:
def tokenize_inputs(example):
    start_prompt = "Summarize the following conversation.\n\n"
    end_prompt = "\n\nSummary: "
    prompt = [start_prompt + dialogue + end_prompt for dialogue in example['dialogue']]
    tokenized_prompt = tokenizer(prompt, padding='max_length', truncation=True, return_tensors='pt', max_length=512)
    tokenized_summary = tokenizer(example['summary'], padding='max_length', truncation=True, return_tensors='pt', max_length=512)

    example['input_ids'] = tokenized_prompt['input_ids']
    example['attention_mask'] = tokenized_prompt['attention_mask']
    example['labels'] = tokenized_summary['input_ids']

    return example

tokenizer.pad_token = tokenizer.eos_token
tokenized_datasets = dataset.map(tokenize_inputs, batched=True)
tokenized_datasets = tokenized_datasets.remove_columns(['id', 'dialogue', 'summary'])
tokenized_datasets = tokenized_datasets.filter(lambda example, index: index % 100 == 0, with_indices=True)

In [ ]:
print(f" train dataset : {tokenized_datasets['train'].shape}")
print(f" validation dataset : {tokenized_datasets['validation'].shape}")
print(f" test dataset : {tokenized_datasets['test'].shape}")

len(tokenized_datasets['train']['labels'][0])

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bart-cnn-samsum-finetuned",  # in local directory
    hub_model_id="HFS26/bart-cnn-samsum-finetune-model",  # identifier on the Hub
    learning_rate=1e-5,
    num_train_epochs=1,
    weight_decay=0.01,
    auto_find_batch_size=True,
    eval_strategy='epoch',
    logging_steps=10
)

trainer = Trainer(
    model=model,
    #tokenizer=tokenizer,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation']
)

In [ ]:
trainer.train()

###push model to Hub

In [ ]:
trainer.push_to_hub()

###Test Model

In [ ]:
loaded_model = AutoModelForSeq2SeqLM.from_pretrained("HFS26/bart-cnn-samsum-finetune-model")
output = generate_summary(sample_data, llm=loaded_model)

print("Sample : ")
print(sample_data)
print("-------------------")
print("Summary :")
print(output)
print("Label :")
print(label)